# CheckEval Qwen3.5-4B Unsloth GRPO

Colab entrypoint for self-checklist judge GRPO. Data should come from HuggingFace Dataset or Google Drive; checkpoints are written to Google Drive so a free Colab runtime can be interrupted safely.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"
        _pil = "pillow"
    !uv pip install -qqq "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install -qqq --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install -qqq transformers==5.2.0 datasets peft accelerate huggingface_hub


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_ROOT = '/content/drive/MyDrive/CheckEval/grpo_runs/qwen35_4b_unsloth'
!mkdir -p "$RUN_ROOT"
print('RUN_ROOT =', RUN_ROOT)

In [ ]:
# Clone the project. If the repo is private, set GITHUB_TOKEN in Colab secrets first.
import os
from google.colab import userdata
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '') or userdata.get('GITHUB_TOKEN') or ''
repo_url = 'github.com/Alex-ZhouZiheng/CheckEval-guided-Fine-tuning.git'
clone_url = f'https://{GITHUB_TOKEN}@{repo_url}' if GITHUB_TOKEN else f'https://{repo_url}'
!rm -rf /content/Thesis
!git clone --depth 1 "$clone_url" /content/Thesis
%cd /content/Thesis

In [ ]:
# Compatibility shim for optional deps that still import the removed Transformers 4.x cache symbol.
!mkdir -p /content/checkeval_compat
with open('/content/checkeval_compat/sitecustomize.py', 'w') as f:
    f.write('''\nimport os\ntry:\n    from transformers.utils import hub\n    if not hasattr(hub, "TRANSFORMERS_CACHE"):\n        hub.TRANSFORMERS_CACHE = os.environ.get("TRANSFORMERS_CACHE", os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface/hub")))\nexcept Exception:\n    pass\n''')
os.environ['PYTHONPATH'] = '/content/checkeval_compat:/content/Thesis/src:' + os.environ.get('PYTHONPATH', '')
print(os.environ['PYTHONPATH'])

In [ ]:
# Option A: download data from HuggingFace Dataset uploaded from the server.
# Set HF_TOKEN in Colab secrets if the dataset repo is private.
from huggingface_hub import hf_hub_download, login
from google.colab import userdata
import os

HF_TOKEN = os.environ.get('HF_TOKEN', '') or userdata.get('HF_TOKEN') or ''
if HF_TOKEN:
    login(token=HF_TOKEN)

HF_DATASET_REPO = 'Alexz1a/checkeval-grpo-data'
DATASET_PATH = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    filename='grpo_train_tier_10k_selfcheck.jsonl',
    token=HF_TOKEN or None,
)
print(DATASET_PATH)

In [ ]:
# Option B: if you copied the JSONL to Drive manually, uncomment this instead.
# DATASET_PATH = '/content/drive/MyDrive/CheckEval/data/grpo_train_tier_10k_selfcheck.jsonl'
# print(DATASET_PATH)

In [ ]:
import os
os.environ['DATASET_PATH'] = DATASET_PATH
os.environ['OUTPUT_DIR'] = RUN_ROOT
os.environ['MODEL_NAME'] = 'unsloth/Qwen3.5-4B'
os.environ['MAX_SAMPLES'] = '500'
os.environ['MAX_STEPS'] = '50'
os.environ['SAVE_STEPS'] = '10'
os.environ['EVAL_STEPS'] = '0'
os.environ['NO_FINAL_EVAL'] = 'true'
os.environ['EVAL_MAX_SAMPLES'] = '50'
os.environ['REWARD_FUNCS'] = 'winner'

!PYTHONPATH="$PYTHONPATH" bash src/train/run_judge_grpo_unsloth.sh

In [ ]:
# Resume / expand after the smoke run passes.
# os.environ['MAX_SAMPLES'] = '10000'
# os.environ['MAX_STEPS'] = '300'
# os.environ['SAVE_STEPS'] = '25'
# os.environ['NO_FINAL_EVAL'] = 'false'
# os.environ['EVAL_MAX_SAMPLES'] = '200'
# !PYTHONPATH="$PYTHONPATH" bash src/train/run_judge_grpo_unsloth.sh